# 12.7 大小差很多的能量怎麼顯示？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：從聽到的起伏，找到每一小段的頻率**

frame→window→STFT→mel→log，每次只加一個步驟。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/audio.svg")))

**先想一想：**把整段音量乘2，會讓log-mel每格乘2嗎？

功率跨度很大，直接看會被最強頻率支配。先clamp避免log0，再取log壓縮動態範圍。單位和底數要與訓練一致。

**把直覺寫成數學：**log(max(power,ε))；amplitude乘2會讓power乘4。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import tone, log_mel

wave = tone()
a = log_mel(wave)
b = log_mel(2 * wave)
print(a.shape, "finite", a.isfinite().all().item(), "平均差", (b - a).mean().item())
assert a.isfinite().all() and b.isfinite().all()

**如何讀結果：**低於clamp的格不符合統一平移；不要把靜音邊緣視爲真實頻率。

**只改一件事：**只把epsilon擴大，說明哪些小能量格受影響。
